In [1]:
import sys
import os

# Change to project directory
os.chdir('/mnt/c/Users/NextGenn/Research/PP/gw-grb-joint-analysis')
sys.path.insert(0, os.getcwd())

# Test imports
import pandas as pd
from src import simulate_gw, simulate_grb, coincidence, statistics
from src.config import DELTA_T, DELTA_OMEGA

print("✓ All imports successful!")
print(f"Working directory: {os.getcwd()}")

✓ All imports successful!
Working directory: /mnt/c/Users/NextGenn/Research/PP/gw-grb-joint-analysis


In [2]:
# Step 1: Generate Simulated Data
gw_data = simulate_gw.simulate_gw_events(n=100)
grb_data = simulate_grb.simulate_grb_events(n=50)

print(f"✓ Generated {len(gw_data)} GW events")
print(f"✓ Generated {len(grb_data)} GRB events")
print("\nGW Data Sample:")
print(gw_data.head())

✓ Generated 100 GW events
✓ Generated 50 GRB events

GW Data Sample:
       time_gps        snr        ra       dec
0  1.126000e+09   6.307655  3.949291  0.321389
1  1.126000e+09   7.961174  1.018527  0.537668
2  1.126000e+09  10.195909  5.864021  0.096332
3  1.126000e+09   8.520240  5.999815 -0.108059
4  1.126000e+09   9.293406  1.639353 -1.251042


In [3]:
# Step 2: Find Coincidences
coinc = coincidence.find_coincidences(gw_data, grb_data, DELTA_T, DELTA_OMEGA)
print(f"✓ Found {len(coinc)} coincidences!")
print("\nCoincidences:")
for i, c in enumerate(coinc[:5]):
    print(f"  {i+1}. GW time: {c[0]:.1f}, GRB time: {c[1]:.1f}, angle: {c[2]:.4f} rad")


✓ Found 0 coincidences!

Coincidences:


In [4]:
# Step 3: Compute Ranking Statistics
if len(coinc) > 0:
    ranking = statistics.ranking_statistic(1.0, 10.0)
    print(f"✓ Ranking statistic (example): {ranking:.4f}")
else:
    print("No coincidences to rank")

No coincidences to rank


In [5]:
# run in your notebook cell
from src.config import DELTA_T, DELTA_OMEGA
print("DELTA_T (s):", DELTA_T)
print("DELTA_OMEGA (rad):", DELTA_OMEGA)
print("GW time range:", gw_data.time_gps.min(), gw_data.time_gps.max())
print("GRB time range:", grb_data.time_gps.min(), grb_data.time_gps.max())

DELTA_T (s): 5.0
DELTA_OMEGA (rad): 0.2
GW time range: 1126000130.3792202 1126009928.2184062
GRB time range: 1126000487.042008 1126009966.9789882


In [6]:
import numpy as np

# time diffs (absolute) between all pairs
tdiff = np.abs(gw_data.time_gps.values[:,None] - grb_data.time_gps.values[None,:])
min_dt = tdiff.min()
print("Min time difference (s):", min_dt)

# angular separations (using coincidence.angular_distance if available)
from src.coincidence import angular_distance
ra_g, dec_g = gw_data.ra.values, gw_data.dec.values
ra_r, dec_r = grb_data.ra.values, grb_data.dec.values
# compute pairwise angles
ang = angular_distance(ra_g[:,None], dec_g[:,None], ra_r[None,:], dec_r[None,:])
min_ang = ang.min()
print("Min angular separation (rad):", min_ang, "≈", np.degrees(min_ang), "deg")

Min time difference (s): 1.5918900966644287
Min angular separation (rad): 0.017587860970182195 ≈ 1.0077102042543051 deg


In [12]:
# try larger search windows
coinc_wider = coincidence.find_coincidences(gw_data, grb_data, dt=30.0, domega=0.5)
len(coinc_wider), coinc_wider[:5]
# optionally write to CSV
import pandas as pd
pd.DataFrame(coinc_wider, columns=["gw_time","grb_time","angle_rad"]).to_csv("data/coincidences_debug.csv", index=False)


In [11]:
# find index of nearest pair
idx = np.unravel_index(np.argmin(tdiff + 1e6*ang), tdiff.shape)  # prioritize time then angle
i_g, i_r = idx
print("Nearest pair -> GW index:", i_g, "GRB index:", i_r)
print(gw_data.iloc[i_g].to_dict())
print(grb_data.iloc[i_r].to_dict())
print("time diff, angle (s, rad):", tdiff[i_g,i_r], ang[i_g,i_r])

Nearest pair -> GW index: 83 GRB index: 32
{'time_gps': 1126008532.74169, 'snr': 14.954696436259102, 'ra': 0.5370698190033659, 'dec': -1.5645920486186382}
{'time_gps': 1126006699.7976537, 'fluence': 0.0005234421828881198, 'ra': 6.2644163600721825, 'dec': -1.5482455023698747}
time diff, angle (s, rad): 1832.944036245346 0.017587860970182195
